In [1]:
import pandas as pd
import numpy as np
import re


In [2]:
df=pd.read_csv(r"C:\Users\USER\Downloads\IMDB Dataset.csv")

In [3]:
print(df.head(2))

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None


In [5]:
print(df.isnull().sum())

review       0
sentiment    0
dtype: int64


In [6]:
print(df.duplicated().sum())

418


In [7]:
df.drop_duplicates(keep="first")



,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [32]:
print(df['new_sent'].value_counts())

new_sent
1    25000
0    25000
Name: count, dtype: int64


In [8]:
import sys
print(sys.executable)
# It shows which Python interpreter your notebook is currently using.
# sometime lib not happen properly use this path then install
# ->path -m pip install lib

c:\Users\USER\OneDrive\Desktop\Python\fashion\myenv\Scripts\python.exe


In [9]:
import spacy 

In [10]:
nlp=spacy.load("en_core_web_sm")


In [11]:
doc=nlp(df["sentiment"].iloc[0])

In [12]:
def preprocess(text):
    text=text.lower()

    text=re.sub(r'<.*?>','',text)
    
    doc=nlp(text)

    clean=[]

    for token in doc:
        if  token.is_punct or  token.is_space:
            continue
        
        if token.is_stop and token.text not in ["not", "no", "never"]:
            continue

        clean.append(token.lemma_)
    
    clean_string=" ".join(clean)

    clean_string=clean_string.strip()

    return clean_string




In [13]:
df['new_sent']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [14]:
df["CleanReviw"]=df['review'].apply(lambda x:preprocess(x))

In [15]:
df['CleanReviw'].iloc[0:3]

0    reviewer mention watch 1 oz episode hook right...
1    wonderful little production filming technique ...
2    think wonderful way spend time hot summer week...
Name: CleanReviw, dtype: object

In [16]:
# import gensim.downloader as api
# by this i can download pretrain model of gensim

In [17]:
from gensim.models import Word2Vec

In [18]:
token=df['CleanReviw'].apply(lambda x:x.split())
# it req for word2vec by gensim


In [19]:
vec = Word2Vec(
    sentences=token,
    vector_size=300,
    window=5,
    min_count=1,
    workers=4
)

In [20]:
def makevec(text):
    vector=[]
    words=text.split()

    for word in words:
      if word in vec.wv:
        # wv word vector
        vector.append(vec.wv[word])
    
    if len(vector)==0:
       return np.zeros(vec.vector_size)
    
    else:
       return np.mean(vector,axis=0)


# THIS MANUAL DESIGN FUNCTION

In [21]:
def getvec(text):
    words=text.split()

    return vec.wv.get_mean_vector(words)

# this done by auto 



PROBLEM WITH WORDTOVEC THIS CAN'T HENDLE OUT OF VOCAB WORD THAT'S THE MAJOR ISSUE

In [22]:
df['word2vec']=df['CleanReviw'].apply(makevec)

In [23]:
df['word2vec'].iloc[:3]

0    [0.029516561, 0.60485476, 0.100154966, -0.1521...
1    [0.06568278, 0.39472598, 0.0020113115, -0.4017...
2    [0.051158506, 0.41340637, 0.23477304, -0.34433...
Name: word2vec, dtype: object

PROBLEM OF OUT OF VOCAB SOLVE BY FASTTEXT

Word2Vec learns embeddings only for complete words present in the training vocabulary, which creates the Out Of Vocabulary (OOV) problem for unseen or misspelled words. FastText overcomes this limitation by learning character-level subword embeddings. Because FastText understands smaller parts of words, it can generate vectors even for unknown or rare words. This makes FastText more robust and often more effective for real-world text data such as reviews, social media posts, and sentiment analysis tasks.

In [24]:
import fasttext

In [25]:
# fast text also has pre trained model we can also directly import that fasttext.load()
# object.get_word_vector(word)

For training FastText on our own dataset and creating embeddings for our custom text data, the official FastText library does not directly work with pandas columns, dataframes, or Python lists. It expects the training data in the form of a text file where each line represents one sentence or review. Therefore, before training the model, we first save the cleaned text data into a .txt file and then provide that file to the FastText training function.

In [26]:
df['CleanReviw'].to_csv("review.txt",
                    index=False,
                    header=False    
                        )

In [27]:
model=fasttext.train_unsupervised(
    "review.txt" ,
     model='skipgram',
     dim=300
)

In [30]:
model.save_model("embeddingModel.bin")
# bin because fastText internally stores binary data.

In [28]:
def makevec(text):
    vector=[]
    doc=nlp(text)
    for token in doc:
        vector.append(model.get_word_vector(token.text))
    
    return np.mean(vector,axis=0)


In [31]:
df['fastvec']=df['CleanReviw'].apply(makevec)

KeyboardInterrupt: 

In [ ]:
df['fastvec'].iloc[0].shape

(300,)

In [ ]:
df

,review,sentiment,new_sent,CleanReviw,word2vec,fastvec
0,One of the other reviewers has mentioned that ...,positive,1,reviewer mention watch 1 oz episode hook right...,"[-0.026699426, 0.457719, 0.056248873, -0.15962...","[-0.03584729, -0.072228484, -0.035713624, -0.1..."
1,A wonderful little production. <br /><br />The...,positive,1,wonderful little production filming technique ...,"[0.0033222826, 0.3704268, 0.046833597, -0.4067...","[-0.0066883173, -0.07709535, -0.012734795, -0...."
2,I thought this was a wonderful way to spend ti...,positive,1,think wonderful way spend time hot summer week...,"[0.034489926, 0.36251265, 0.20206617, -0.38841...","[-0.01983388, -0.087059036, 0.025548121, -0.18..."
3,Basically there's a family where a little boy ...,negative,0,basically family little boy jake think zombie ...,"[-0.08764966, 0.41330588, 0.09767992, -0.40986...","[-0.058101222, -0.0809275, 0.023560697, -0.155..."
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1,petter mattei love time money visually stunnin...,"[0.09053255, 0.37841687, 0.14862064, -0.409492...","[-0.020965979, -0.03894619, -0.00081404706, -0..."
...,...,...,...,...,...,...
49995,I thought this movie did a down right good job...,positive,1,think movie right good job creative original e...,"[-0.10281654, 0.24571796, 0.12336803, -0.52412...","[-0.055000912, -0.07964227, 0.0059318366, -0.2..."
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,0,bad plot bad dialogue bad acting idiotic direc...,"[-0.009588385, 0.14800201, 0.0387686, -0.31179...","[-0.03907632, -0.084842786, -0.0104280375, -0...."
49997,I am a Catholic taught in parochial elementary...,negative,0,catholic teach parochial elementary school nun...,"[0.06726641, 0.59463215, 0.17383334, -0.293474...","[-0.037094742, 0.02060916, -0.039100826, -0.17..."
49998,I'm going to have to disagree with the previou...,negative,0,go disagree previous comment maltin second rat...,"[0.037347194, 0.38640636, 0.043965325, -0.2424...","[0.0024010646, -0.040684387, -0.024939364, -0...."


In [ ]:
import sklearn

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df['fastvec'],df['new_sent'],test_size=0.2,random_state=42,stratify=df['new_sent'])

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras  import layers


In [ ]:
dlmodel=Sequential(
    [
        layers.Dense(128,activation='relu',input_shape=(300,)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(64,activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32,activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.25),
        layers.Dense(1,activation='sigmoid'),
        
    ]
)

c:\Users\USER\OneDrive\Desktop\Python\fashion\myenv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
dlmodel.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy'],

)

In [ ]:
X_train.dtypes

dtype('O')

In [ ]:
print(type(X_train.iloc[0]))  #each row  is one vector 
print(X_train.iloc[0])

<class 'numpy.ndarray'>
[-2.69520888e-03 -2.82191020e-02  3.46727706e-02 -1.84085235e-01
  7.67316595e-02 -2.41399426e-02  1.44793838e-01  2.62031984e-02
  9.95901413e-03  3.55569869e-02 -4.82561179e-02  2.80970111e-02
  5.58379143e-02 -6.03941791e-02 -1.21039394e-02 -5.71627542e-02
 -2.83049475e-02 -3.68165523e-02 -1.98469702e-02 -6.81091323e-02
 -2.38628939e-01 -6.56989068e-02  1.00552954e-01 -4.50814515e-02
 -8.65867808e-02 -4.17422540e-02 -4.61335778e-02 -1.08987071e-01
  5.79239018e-02 -1.28258571e-01 -1.74804598e-01  1.25698403e-01
 -1.18246958e-01 -1.65004339e-02  1.73392135e-03 -5.63416108e-02
 -6.04676045e-02 -9.98339709e-03 -4.38001417e-02 -8.53305012e-02
  6.39385486e-04  2.92683374e-02  3.37645039e-03 -2.17294022e-02
  1.23975933e-01  9.79449973e-02  6.95192665e-02  7.23599270e-02
  1.47441775e-01  2.33530909e-01 -8.44099671e-02 -1.08308196e-02
 -8.71825293e-02 -1.77887842e-01  9.44899768e-03 -1.44717380e-01
 -2.51691714e-02 -3.78146097e-02  1.77237839e-01  1.06712341e-01
 

In [ ]:
x_train = np.vstack(X_train).astype(np.float32)
x_test = np.vstack(X_test).astype(np.float32)

y_train = np.array(y_train, dtype=np.float32)
y_test = np.array(y_test, dtype=np.float32)

# Each row is an independent NumPy vector.
# But deep learning models expect one single 2D matrix:


In [ ]:
dlmodel.fit(x_train,y_train,epochs=60,batch_size=32)

Epoch 1/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.8345 - loss: 0.3754
Epoch 2/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8622 - loss: 0.3302
Epoch 3/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8643 - loss: 0.3252
Epoch 4/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8639 - loss: 0.3240
Epoch 5/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8674 - loss: 0.3163
Epoch 6/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8633 - loss: 0.3217
Epoch 7/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8650 - loss: 0.3229
Epoch 8/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8648 - loss: 0.3214
Epoch 9/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8668 - loss: 0.3196
Epoch 10/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8666 - loss: 0.3158
Epoch 11/60
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8706 - loss: 0.3156
Epoch 12/60
1250/1250 ━━━━━━━━

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix

In [ ]:
y_pred=list((dlmodel.predict(x_test)>0.5).astype(int))

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [ ]:
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.86      0.91      0.88      4727
           1       0.91      0.86      0.89      5273

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



In [ ]:
print(confusion_matrix(y_pred,y_test))

[[4280  447]
 [ 720 4553]]


In [ ]:
# dlmodel.save("ReviewAnalyiser.h5")